In [ ]:
"""
Ambil SEMUA data historis location 6144741 (STT Satyabhakti, Malang)
dari OpenAQ v3 — semua parameter, semua halaman
"""

import requests, pandas as pd, time
from datetime import datetime, timezone
from pathlib import Path

API_KEY  = "08dbcf19493a5142fec8541351d25d6049772d88d766ffc8f8454952a06fde64"
BASE_URL = "https://api.openaq.org/v3"
HEADERS  = {"X-API-Key": API_KEY}
LOC_ID   = 6144741
OUT_DIR  = Path("openaq_malang")
OUT_DIR.mkdir(exist_ok=True)

def safe(d, *keys, default=""):
    for k in keys:
        if not isinstance(d, dict): return default
        d = d.get(k)
        if d is None: return default
    return d if d is not None else default

def rget(url, params=None, retries=4):
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if r.status_code == 429:
                wait = 60 * (attempt + 1)
                print(f"    ⏳ Rate limit, tunggu {wait}s...")
                time.sleep(wait)
                continue
            if r.status_code != 200:
                print(f"    ⚠️  HTTP {r.status_code}")
                return None
            return r.json()
        except Exception as e:
            print(f"    ⚠️  Error: {e}")
            time.sleep(5)
    return None

# ══════════════════════════════════════════════════════════
# STEP 1 — INFO LOKASI & SENSOR
# ══════════════════════════════════════════════════════════
print("\n" + "═"*55)
print(f"STEP 1: Info lokasi {LOC_ID}")
print("═"*55)

loc_data = rget(f"{BASE_URL}/locations/{LOC_ID}")
if loc_data:
    loc = loc_data.get("results", [{}])[0]
    print(f"  Nama     : {loc.get('name')}")
    print(f"  Kota     : {loc.get('locality')}")
    print(f"  Provider : {safe(loc,'provider','name')}")
    print(f"  Data dari: {safe(loc,'datetimeFirst','utc')}")
    print(f"  Data s/d : {safe(loc,'datetimeLast','utc')}")
    date_first = safe(loc, "datetimeFirst", "utc")
    date_last  = safe(loc, "datetimeLast",  "utc")
else:
    print("  ⚠️  Gagal ambil info lokasi")
    date_first = "2024-01-01T00:00:00Z"
    date_last  = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# Ambil sensor ID
print(f"\n  Sensors:")
sensor_data = rget(f"{BASE_URL}/locations/{LOC_ID}/sensors")
sensors = []
if sensor_data:
    for s in sensor_data.get("results", []):
        sid   = s.get("id")
        param = safe(s, "parameter", "name")
        unit  = safe(s, "parameter", "units")
        sensors.append({"sensor_id": sid, "parameter": param, "unit": unit})
        print(f"    sensor_id={sid} | {param} ({unit})")

df_sensors = pd.DataFrame(sensors)
print(f"\n  Total sensor: {len(sensors)}")

# ══════════════════════════════════════════════════════════
# STEP 2 — FETCH SEMUA MEASUREMENTS
# ══════════════════════════════════════════════════════════
print("\n" + "═"*55)
print("STEP 2: Fetch semua data historis")
print(f"        Dari : {date_first}")
print(f"        S/d  : {date_last}")
print("═"*55)

meas_file    = OUT_DIR / f"malang_{LOC_ID}_full.csv"
write_header = True
total_rows   = 0

for idx, row in df_sensors.iterrows():
    sid   = row["sensor_id"]
    param = row["parameter"]
    unit  = row["unit"]

    print(f"\n  📡 Sensor {sid} | {param}")
    rows, page = [], 1

    while True:
        data = rget(f"{BASE_URL}/sensors/{sid}/measurements", params={
            "date_from" : date_first,
            "date_to"   : date_last,
            "limit"     : 1000,
            "page"      : page,
        })
        if not data: break
        results = data.get("results", [])
        if not results: break

        for m in results:
            val = m.get("value")
            if isinstance(val, dict):
                val = val.get("value")
            rows.append({
                "sensor_id"     : sid,
                "location_id"   : LOC_ID,
                "parameter"     : param,
                "unit"          : unit,
                "value"         : val,
                "datetime_utc"  : safe(m,"period","datetimeFrom","utc") or safe(m,"date","utc"),
                "datetime_local": safe(m,"period","datetimeFrom","local") or safe(m,"date","local"),
            })

        print(f"    Hal.{page:>3} | +{len(results):>5} baris | subtotal: {len(rows):,}")
        if len(results) < 1000: break
        page += 1
        time.sleep(0.4)

    if rows:
        pd.DataFrame(rows).to_csv(
            meas_file, mode="a", index=False, header=write_header
        )
        write_header = False
        total_rows  += len(rows)
        print(f"    ✅ {len(rows):,} baris disimpan")

print(f"\n✅ TOTAL {total_rows:,} baris → {meas_file}")

# ══════════════════════════════════════════════════════════
# STEP 3 — PIVOT & RINGKASAN
# ══════════════════════════════════════════════════════════
if total_rows == 0:
    print("\n⚠️  Tidak ada data. Sensor mungkin tidak punya histori lebih panjang.")
    exit()

print("\n" + "═"*55)
print("STEP 3: Pivot harian & ringkasan statistik")
print("═"*55)

df = pd.read_csv(meas_file)
df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], errors="coerce", utc=True)
df["date"]  = df["datetime_utc"].dt.date
df["hour"]  = df["datetime_utc"].dt.hour
df["month"] = df["datetime_utc"].dt.to_period("M").astype(str)

# Pivot harian
df_daily = (
    df.groupby(["date","parameter"])["value"]
    .agg(avg="mean", min="min", max="max", n_obs="count")
    .reset_index().round(3)
)
df_daily.to_csv(OUT_DIR / f"malang_{LOC_ID}_daily.csv", index=False)

# Ringkasan
df_summary = (
    df.groupby("parameter")["value"]
    .agg(mean="mean", median="median", std="std",
         min="min", max="max", n_obs="count")
    .reset_index().round(3)
)
df_summary.to_csv(OUT_DIR / f"malang_{LOC_ID}_summary.csv", index=False)

print(f"\n📅 Rentang data : {df['date'].min()} → {df['date'].max()}")
print(f"📊 Total hari   : {df['date'].nunique()}")
print(f"\nRingkasan per parameter:")
print(df_summary.to_string(index=False))

print("\n" + "═"*55)
print("✅ SELESAI!")
for f in sorted(OUT_DIR.glob(f"*{LOC_ID}*")):
    print(f"   {f.name:<45} {f.stat().st_size/1024:.1f} KB")


═══════════════════════════════════════════════════════
STEP 1: Info lokasi 6144741
═══════════════════════════════════════════════════════
  Nama     : STT Satyabhakti 
  Kota     : None
  Provider : AirGradient
  Data dari: 2025-11-27T16:00:00Z
  Data s/d : 2026-05-27T10:00:00Z

  Sensors:
    sensor_id=14739442 | pm1 (µg/m³)
    sensor_id=14739443 | pm25 (µg/m³)
    sensor_id=14739444 | relativehumidity (%)
    sensor_id=14739445 | temperature (c)
    sensor_id=14739446 | um003 (particles/cm³)

  Total sensor: 5

═══════════════════════════════════════════════════════
STEP 2: Fetch semua data historis
        Dari : 2025-11-27T16:00:00Z
        S/d  : 2026-05-27T10:00:00Z
═══════════════════════════════════════════════════════

  📡 Sensor 14739442 | pm1
    Hal.  1 | + 1000 baris | subtotal: 1,000
    Hal.  2 | + 1000 baris | subtotal: 2,000
    Hal.  3 | + 1000 baris | subtotal: 3,000
    Hal.  4 | + 1000 baris | subtotal: 4,000
    Hal.  5 | +  188 baris | subtotal: 4,188
    ✅ 4

/tmp/ipykernel_11545/1697476113.py:155: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["month"] = df["datetime_utc"].dt.to_period("M").astype(str)


In [ ]:
"""
================================================================
ANALISIS KUALITAS UDARA MALANG — EDA + MODEL ML + EVALUASI
Dataset  : malang_6144741_full.csv
Sensor   : AirGradient IoT, STT Satyabhakti, Malang
Periode  : Nov 2025 – Mei 2026
================================================================
Struktur:
  BAGIAN 1 — Load & Preprocessing
  BAGIAN 2 — EDA (statistik, tren, pola temporal)
  BAGIAN 3 — Analisis Korelasi
  BAGIAN 4 — Feature Engineering
  BAGIAN 5 — Model ML (Linear Reg, Random Forest, XGBoost, LSTM)
  BAGIAN 6 — Evaluasi & Perbandingan Model
  BAGIAN 7 — Simpan hasil
================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

import os
os.makedirs("hasil_analisis", exist_ok=True)

# Warna tema konsisten
COLORS = {
    "pm25"  : "#E63946",
    "pm1"   : "#F4A261",
    "temp"  : "#2A9D8F",
    "rh"    : "#457B9D",
    "um003" : "#6A4C93",
    "accent": "#264653",
}
plt.rcParams.update({
    "figure.dpi"      : 150,
    "font.family"     : "DejaVu Sans",
    "axes.spines.top" : False,
    "axes.spines.right": False,
    "axes.grid"       : True,
    "grid.alpha"      : 0.3,
})

# ================================================================
# BAGIAN 1 — LOAD & PREPROCESSING
# ================================================================
print("="*60)
print("BAGIAN 1: Load & Preprocessing")
print("="*60)

df_raw = pd.read_csv("malang_6144741_full.csv", sep=";")

# --- TAMBAHAN PERBAIKAN FORMAT ANGKA ---
def fix_val(x):
    x = str(x)
    # kalau titik lebih dari satu
    if x.count('.') > 1:
        parts = x.split('.')
        x = parts[0] + '.' + ''.join(parts[1:])

    return float(x)

df_raw['value'] = df_raw['value'].apply(fix_val)
# ---------------------------------------

# Parse waktu UTC
df_raw["datetime_utc"] = pd.to_datetime(df_raw["datetime_utc"], utc=True)

# --- TAMBAHAN PERBAIKAN ZONA WAKTU LOKAL (WIB / MALANG) ---
df_raw["datetime_local"] = df_raw["datetime_utc"].dt.tz_convert('Asia/Jakarta')
# ----------------------------------------------------------

# Pivot: tiap parameter jadi kolom (Gunakan datetime_local agar jamnya akurat!)
df = df_raw.pivot_table(
    index="datetime_local",
    columns="parameter",
    values="value",
    aggfunc="mean"
).reset_index()
df.columns.name = None
df = df.rename(columns={
    "pm1"             : "PM1",
    "pm25"            : "PM25",
    "relativehumidity": "RH",
    "temperature"     : "Temp",
    "um003"           : "UM003",
})
df = df.sort_values("datetime_local").reset_index(drop=True)

# Fitur waktu (Sekarang sudah akurat waktu Malang)
df["hour"]        = df["datetime_local"].dt.hour
df["day_of_week"] = df["datetime_local"].dt.dayofweek   # 0=Senin
df["month"]       = df["datetime_local"].dt.month
df["date"]        = df["datetime_local"].dt.date
df["is_weekend"]  = df["day_of_week"].isin([5,6]).astype(int)
df["time_of_day"] = pd.cut(df["hour"],
    bins=[-1,5,11,17,23],
    labels=["Malam (00-05)","Pagi (06-11)","Siang (12-17)","Sore/Malam (18-23)"]
)

# ==========================================
# CLEANING OUTLIER & SENSOR GLITCH
# ==========================================

# 1. Hapus outlier untuk polutan (menggunakan Quantile 99.5%)
for col in ["PM25","PM1","UM003"]:
    q995 = df[col].quantile(0.995)
    df[col] = df[col].clip(upper=q995)

# 2. Batas Fisika untuk Suhu (Temp) & Kelembaban (RH)
# Suhu di Malang normalnya 15°C - 40°C. Lebih dari itu = Error Sensor
df.loc[(df["Temp"] > 40) | (df["Temp"] < 10), "Temp"] = np.nan

# Kelembaban (RH) secara fisika tidak mungkin lebih dari 100%
df.loc[(df["RH"] > 100) | (df["RH"] < 0), "RH"] = np.nan

# 3. Interpolasi: Isi nilai NaN tadi dengan rata-rata garis lurus dari jam sebelum & sesudahnya
df["Temp"] = df["Temp"].interpolate(method='linear')
df["RH"] = df["RH"].interpolate(method='linear')

# Jika ada NaN di baris paling awal yang gagal di-interpolasi, gunakan backward fill
df.fillna(method='bfill', inplace=True)

print(f"Shape setelah pivot : {df.shape}")
print(f"Rentang waktu       : {df['datetime_local'].min().date()} → {df['datetime_local'].max().date()}")
print(f"Missing values      :\n{df[['PM25','PM1','RH','Temp','UM003']].isna().sum().to_string()}")
print(f"\nStatistik deskriptif:")
print(df[["PM25","PM1","RH","Temp","UM003"]].describe().round(2).to_string())


# ================================================================
# BAGIAN 2 — EDA
# ================================================================
print("\n" + "="*60)
print("BAGIAN 2: Exploratory Data Analysis")
print("="*60)

# ── 2.1 Distribusi semua parameter ──────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Distribusi Parameter Kualitas Udara — Malang 2025–2026",
             fontsize=14, fontweight="bold", y=1.01)

params = [("PM25","PM2.5 (µg/m³)",COLORS["pm25"]),
          ("PM1", "PM1 (µg/m³)",  COLORS["pm1"]),
          ("Temp","Suhu (°C)",     COLORS["temp"]),
          ("RH",  "Kelembaban (%)",COLORS["rh"]),
          ("UM003","Partikel 0.3µm",COLORS["um003"])]

for ax, (col, label, color) in zip(axes.flat, params):
    ax.hist(df[col].dropna(), bins=50, color=color, alpha=0.8, edgecolor="white")
    ax.axvline(df[col].mean(),   color="black", lw=1.5, ls="--", label=f"Mean={df[col].mean():.1f}")
    ax.axvline(df[col].median(), color="gray",  lw=1.5, ls=":",  label=f"Median={df[col].median():.1f}")
    ax.set_title(label, fontweight="bold")
    ax.legend(fontsize=8)

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig("hasil_analisis/01_distribusi_parameter.png", bbox_inches="tight")
plt.close()
print("✅ Saved: 01_distribusi_parameter.png")

# ── 2.2 Tren PM2.5 harian ──────────────────────────────────────
df_daily = df.groupby("date")[["PM25","PM1","Temp","RH"]].mean().reset_index()
df_daily["date"] = pd.to_datetime(df_daily["date"])
df_daily["PM25_7d"] = df_daily["PM25"].rolling(7).mean()

fig, ax = plt.subplots(figsize=(16, 5))
ax.fill_between(df_daily["date"], df_daily["PM25"],
                alpha=0.25, color=COLORS["pm25"])
ax.plot(df_daily["date"], df_daily["PM25"],
        color=COLORS["pm25"], lw=0.8, alpha=0.6, label="PM2.5 Harian")
ax.plot(df_daily["date"], df_daily["PM25_7d"],
        color=COLORS["pm25"], lw=2.5, label="Rata-rata 7 hari")
ax.axhline(15,  color="orange", lw=1.5, ls="--", label="Baku mutu WHO (15 µg/m³)")
ax.axhline(35,  color="red",    lw=1.5, ls="--", label="Baku mutu KLHK (35 µg/m³)")
ax.set_title("Tren PM2.5 Harian — Malang (Nov 2025 – Mei 2026)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Tanggal")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.legend()
plt.tight_layout()
plt.savefig("hasil_analisis/02_tren_pm25_harian.png", bbox_inches="tight")
plt.close()
print("✅ Saved: 02_tren_pm25_harian.png")

# ── 2.3 Pola per jam (diurnal) ─────────────────────────────────
df_hourly = df.groupby("hour")[["PM25","PM1","Temp","RH"]].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Pola Rata-rata per Jam (Diurnal)", fontsize=13, fontweight="bold")

for ax, (col, label, color) in zip(axes.flat, [
    ("PM25","PM2.5 (µg/m³)",COLORS["pm25"]),
    ("PM1", "PM1 (µg/m³)",  COLORS["pm1"]),
    ("Temp","Suhu (°C)",     COLORS["temp"]),
    ("RH",  "Kelembaban (%)",COLORS["rh"]),
]):
    ax.fill_between(df_hourly.index, df_hourly[col],
                    alpha=0.2, color=color)
    ax.plot(df_hourly.index, df_hourly[col], color=color, lw=2.5, marker="o", ms=4)
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("Jam")
    ax.set_xticks(range(0,24,3))

plt.tight_layout()
plt.savefig("hasil_analisis/03_pola_diurnal.png", bbox_inches="tight")
plt.close()
print("✅ Saved: 03_pola_diurnal.png")

# ── 2.4 Pola per hari dalam seminggu ──────────────────────────
days_label = ["Sen","Sel","Rab","Kam","Jum","Sab","Min"]
df_dow = df.groupby("day_of_week")["PM25"].mean()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(days_label, df_dow.values,
              color=[COLORS["pm25"] if i<5 else COLORS["rh"] for i in range(7)],
              edgecolor="white", linewidth=0.5)
ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=9)
ax.set_title("Rata-rata PM2.5 per Hari dalam Seminggu", fontsize=13, fontweight="bold")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.set_ylim(0, df_dow.max() * 1.2)
plt.tight_layout()
plt.savefig("hasil_analisis/04_pola_harian_mingguan.png", bbox_inches="tight")
plt.close()
print("✅ Saved: 04_pola_harian_mingguan.png")

# ── 2.5 Boxplot per waktu ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
df.boxplot(column="PM25", by="time_of_day", ax=ax,
           patch_artist=True,
           boxprops=dict(facecolor=COLORS["pm25"], alpha=0.5),
           medianprops=dict(color="black", lw=2))
ax.set_title("Distribusi PM2.5 per Periode Waktu", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("PM2.5 (µg/m³)")
plt.suptitle("")
plt.tight_layout()
plt.savefig("hasil_analisis/05_boxplot_waktu.png", bbox_inches="tight")
plt.close()
print("✅ Saved: 05_boxplot_waktu.png")

# ── 2.6 Tren bulanan ──────────────────────────────────────────
df["month_label"] = df["datetime_local"].dt.strftime("%b %Y")
df_monthly = df.groupby(["month","month_label"])["PM25"].agg(["mean","std"]).reset_index()
df_monthly = df_monthly.sort_values("month")

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(df_monthly["month_label"], df_monthly["mean"],
       yerr=df_monthly["std"], capsize=4,
       color=COLORS["pm25"], alpha=0.75, edgecolor="white")
ax.axhline(15, color="orange", lw=1.5, ls="--", label="WHO (15 µg/m³)")
ax.axhline(35, color="red",    lw=1.5, ls="--", label="KLHK (35 µg/m³)")
ax.set_title("Rata-rata PM2.5 per Bulan ± Std Dev", fontsize=13, fontweight="bold")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.legend()
plt.tight_layout()
plt.savefig("hasil_analisis/06_tren_bulanan.png", bbox_inches="tight")
plt.close()
print("✅ Saved: 06_tren_bulanan.png")


# ================================================================
# BAGIAN 3 — KORELASI
# ================================================================
print("\n" + "="*60)
print("BAGIAN 3: Analisis Korelasi")
print("="*60)

feat_cols = ["PM25","PM1","UM003","Temp","RH"]
corr = df[feat_cols].corr()
print(corr.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap korelasi
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, vmin=-1, vmax=1,
            ax=axes[0], linewidths=0.5, square=True)
axes[0].set_title("Matriks Korelasi", fontsize=13, fontweight="bold")

# Scatter PM1 vs PM25
axes[1].scatter(df["PM1"], df["PM25"], alpha=0.15,
                color=COLORS["pm25"], s=10)
corr_df = df[["PM1","PM25"]].dropna()
m, b = np.polyfit(corr_df["PM1"], corr_df["PM25"], 1)
x_line = np.linspace(df["PM1"].min(), df["PM1"].max(), 100)
axes[1].plot(x_line, m*x_line+b, color="black", lw=2, label=f"y={m:.2f}x+{b:.2f}")
axes[1].set_xlabel("PM1 (µg/m³)")
axes[1].set_ylabel("PM2.5 (µg/m³)")
axes[1].set_title(f"Scatter PM1 vs PM2.5  (r={corr.loc['PM1','PM25']:.3f})",
                  fontsize=13, fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig("hasil_analisis/07_korelasi.png", bbox_inches="tight")
plt.close()
print("✅ Saved: 07_korelasi.png")


# ================================================================
# BAGIAN 4 — FEATURE ENGINEERING
# ================================================================
print("\n" + "="*60)
print("BAGIAN 4: Feature Engineering")
print("="*60)

df = df.sort_values("datetime_local").reset_index(drop=True)

# Lag features (1, 2, 3, 6, 12, 24 jam)
for lag in [1, 2, 3, 6, 12, 24]:
    df[f"PM25_lag{lag}"] = df["PM25"].shift(lag)

# Rolling statistics
for win in [3, 6, 12, 24]:
    df[f"PM25_roll{win}_mean"] = df["PM25"].shift(1).rolling(win).mean()
    df[f"PM25_roll{win}_std"]  = df["PM25"].shift(1).rolling(win).std()

# Cyclic encoding waktu
df["hour_sin"]  = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"]  = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"]   = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"]   = np.cos(2 * np.pi * df["day_of_week"] / 7)

df = df.dropna().reset_index(drop=True)
print(f"Shape setelah feature engineering: {df.shape}")

FEATURE_COLS = (
    ["PM1","UM003","Temp","RH"] +
    [f"PM25_lag{l}" for l in [1,2,3,6,12,24]] +
    [f"PM25_roll{w}_mean" for w in [3,6,12,24]] +
    [f"PM25_roll{w}_std"  for w in [3,6,12,24]] +
    ["hour_sin","hour_cos","dow_sin","dow_cos","is_weekend"]
)
TARGET = "PM25"

X = df[FEATURE_COLS].values
y = df[TARGET].values

# Split kronologis (80/20)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(f"Train: {len(X_train)} | Test: {len(X_test)}")


# ================================================================
# BAGIAN 5 — MODEL ML
# ================================================================
print("\n" + "="*60)
print("BAGIAN 5: Training Model ML")
print("="*60)

results   = {}
preds_all = {}

def evaluate(name, y_true, y_pred):
    mae   = mean_absolute_error(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    r2    = r2_score(y_true, y_pred)
    mask = y_true > 1
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    print(f"  {name:<25} MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}  MAPE={mape:.2f}%")
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}

# ── Model 1: Linear Regression ──────────────────────────────
print("\n[1] Linear Regression")
scaler = MinMaxScaler()
X_tr_s = scaler.fit_transform(X_train)
X_te_s = scaler.transform(X_test)
lr = LinearRegression()
lr.fit(X_tr_s, y_train)
p_lr = lr.predict(X_te_s)
results["Linear Regression"]  = evaluate("Linear Regression", y_test, p_lr)
preds_all["Linear Regression"] = p_lr

# ── Model 2: Random Forest ───────────────────────────────────
print("\n[2] Random Forest")
rf = RandomForestRegressor(n_estimators=200, max_depth=15,
                            min_samples_leaf=2, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
p_rf = rf.predict(X_test)
results["Random Forest"]  = evaluate("Random Forest", y_test, p_rf)
preds_all["Random Forest"] = p_rf

# ── Model 3: Gradient Boosting (XGBoost-like) ────────────────
print("\n[3] Gradient Boosting")
gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                max_depth=5, subsample=0.8,
                                min_samples_leaf=2, random_state=42)
gb.fit(X_train, y_train)
p_gb = gb.predict(X_test)
results["Gradient Boosting"]  = evaluate("Gradient Boosting", y_test, p_gb)
preds_all["Gradient Boosting"] = p_gb

# ── Model 4: LSTM ────────────────────────────────────────────
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    print("\n[4] LSTM")

    LOOKBACK = 24
    scaler_lstm = MinMaxScaler()
    y_scaled    = scaler_lstm.fit_transform(y.reshape(-1,1)).flatten()
    X_scaled    = MinMaxScaler().fit_transform(X)

    def make_sequences(X, y, lookback):
        Xs, ys = [], []
        for i in range(lookback, len(X)):
            Xs.append(X[i-lookback:i])
            ys.append(y[i])
        return np.array(Xs), np.array(ys)

    X_seq, y_seq = make_sequences(X_scaled, y_scaled, LOOKBACK)
    split_lstm   = int(len(X_seq) * 0.8)
    Xl_tr, Xl_te = X_seq[:split_lstm], X_seq[split_lstm:]
    yl_tr, yl_te = y_seq[:split_lstm], y_seq[split_lstm:]

    model_lstm = Sequential([
        LSTM(64, return_sequences=True, input_shape=(LOOKBACK, X.shape[1])),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(1)
    ])
    model_lstm.compile(optimizer="adam", loss="mse")
    es = EarlyStopping(patience=5, restore_best_weights=True)
    model_lstm.fit(Xl_tr, yl_tr, epochs=30, batch_size=64,
                   validation_split=0.1, callbacks=[es], verbose=0)

    p_lstm_s = model_lstm.predict(Xl_te, verbose=0).flatten()
    p_lstm   = scaler_lstm.inverse_transform(p_lstm_s.reshape(-1,1)).flatten()
    y_lstm   = scaler_lstm.inverse_transform(yl_te.reshape(-1,1)).flatten()

    results["LSTM"]  = evaluate("LSTM", y_lstm, p_lstm)
    preds_all["LSTM"] = (p_lstm, y_lstm)
    HAS_LSTM = True
    print("  ✅ LSTM selesai")

except ImportError:
    print("\n[4] LSTM — TensorFlow tidak tersedia, skip")
    HAS_LSTM = False


# ================================================================
# BAGIAN 6 — EVALUASI & VISUALISASI
# ================================================================
print("\n" + "="*60)
print("BAGIAN 6: Evaluasi & Visualisasi Hasil")
print("="*60)

# Tabel perbandingan
df_results = pd.DataFrame(results).T.reset_index()
df_results.columns = ["Model","MAE","RMSE","R2","MAPE"]
df_results = df_results.sort_values("RMSE")
print("\nRingkasan Perbandingan Model:")
print(df_results.round(4).to_string(index=False))
df_results.to_csv("hasil_analisis/model_comparison.csv", index=False)

# ── Bar chart perbandingan metrik ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Perbandingan Performa Model ML", fontsize=14, fontweight="bold")
palette = ["#2A9D8F","#E9C46A","#F4A261","#E76F51"]

for ax, metric in zip(axes, ["MAE","RMSE","R2"]):
    bars = ax.bar(df_results["Model"], df_results[metric],
                  color=palette[:len(df_results)], edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
    ax.set_title(metric, fontweight="bold")
    ax.set_xticklabels(df_results["Model"], rotation=15, ha="right")
    if metric == "R2":
        ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig("hasil_analisis/08_perbandingan_model.png", bbox_inches="tight")
plt.close()
print("✅ Saved: 08_perbandingan_model.png")

# ── Actual vs Predicted (best model) ────────────────────────
best_model = df_results.iloc[0]["Model"]
print(f"\nBest model: {best_model}")

if best_model in ["Random Forest","Gradient Boosting","Linear Regression"]:
    y_pred_best = preds_all[best_model]
    y_true_best = y_test
elif best_model == "LSTM" and HAS_LSTM:
    y_pred_best, y_true_best = preds_all["LSTM"]

# Time series plot
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

n_show = min(500, len(y_pred_best))
axes[0].plot(y_true_best[:n_show], color=COLORS["accent"], lw=1.5,
             label="Aktual", alpha=0.9)
axes[0].plot(y_pred_best[:n_show], color=COLORS["pm25"], lw=1.5,
             ls="--", label="Prediksi", alpha=0.9)
axes[0].set_title(f"Aktual vs Prediksi PM2.5 — {best_model} (500 sampel pertama)",
                  fontsize=13, fontweight="bold")
axes[0].set_ylabel("PM2.5 (µg/m³)")
axes[0].legend()

# Scatter actual vs pred
axes[1].scatter(y_true_best, y_pred_best, alpha=0.25,
                color=COLORS["pm25"], s=10)
lim = max(y_true_best.max(), y_pred_best.max())
axes[1].plot([0,lim],[0,lim], "k--", lw=1.5, label="Ideal (y=x)")
axes[1].set_xlabel("PM2.5 Aktual (µg/m³)")
axes[1].set_ylabel("PM2.5 Prediksi (µg/m³)")
axes[1].set_title(f"Scatter Aktual vs Prediksi — {best_model}", fontsize=13, fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig("hasil_analisis/09_aktual_vs_prediksi.png", bbox_inches="tight")
plt.close()
print("✅ Saved: 09_aktual_vs_prediksi.png")

# ── Feature Importance ────────────────────────────────────
if best_model == "Random Forest":
    imp = pd.Series(rf.feature_importances_, index=FEATURE_COLS).nlargest(15)
elif best_model == "Gradient Boosting":
    imp = pd.Series(gb.feature_importances_, index=FEATURE_COLS).nlargest(15)
else:
    imp = None

if imp is not None:
    fig, ax = plt.subplots(figsize=(10, 6))
    imp.sort_values().plot.barh(ax=ax, color=COLORS["pm25"], edgecolor="white")
    ax.set_title(f"Top 15 Feature Importance — {best_model}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Importance Score")
    plt.tight_layout()
    plt.savefig("hasil_analisis/10_feature_importance.png", bbox_inches="tight")
    plt.close()
    print("✅ Saved: 10_feature_importance.png")

# ── Residual analysis ─────────────────────────────────────
residuals = y_true_best - y_pred_best
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(residuals, bins=50, color=COLORS["accent"], edgecolor="white", alpha=0.8)
axes[0].axvline(0, color="red", lw=2, ls="--")
axes[0].set_title("Distribusi Residual", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Residual (Aktual - Prediksi)")

axes[1].scatter(y_pred_best, residuals, alpha=0.2, s=10, color=COLORS["accent"])
axes[1].axhline(0, color="red", lw=2, ls="--")
axes[1].set_title("Residual vs Fitted", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Prediksi")
axes[1].set_ylabel("Residual")

plt.tight_layout()
plt.savefig("hasil_analisis/11_residual_analysis.png", bbox_inches="tight")
plt.close()
print("✅ Saved: 11_residual_analysis.png")


# ================================================================
# BAGIAN 7 — SIMPAN HASIL
# ================================================================
print("\n" + "="*60)
print("BAGIAN 7: Simpan Hasil")
print("="*60)

# Dataset final dengan prediksi best model
df_out = df.iloc[split:].copy().reset_index(drop=True)
df_out["PM25_pred"] = y_pred_best[:len(df_out)]
df_out[["datetime_local","PM25","PM1","Temp","RH","UM003","PM25_pred"]].to_csv(
    "hasil_analisis/predictions.csv", index=False
)

print("\n📁 Semua file tersimpan di folder: hasil_analisis/")
print("─"*55)
files = [
    ("01_distribusi_parameter.png",  "Distribusi semua parameter"),
    ("02_tren_pm25_harian.png",      "Tren PM2.5 harian + rolling average"),
    ("03_pola_diurnal.png",          "Pola per jam (diurnal)"),
    ("04_pola_harian_mingguan.png",  "Pola per hari dalam seminggu"),
    ("05_boxplot_waktu.png",         "Distribusi PM2.5 per periode waktu"),
    ("06_tren_bulanan.png",          "Tren bulanan PM2.5"),
    ("07_korelasi.png",              "Heatmap korelasi + scatter PM1 vs PM2.5"),
    ("08_perbandingan_model.png",    "Perbandingan MAE/RMSE/R² semua model"),
    ("09_aktual_vs_prediksi.png",    "Time series & scatter aktual vs prediksi"),
    ("10_feature_importance.png",    "Feature importance best model"),
    ("11_residual_analysis.png",     "Analisis residual"),
    ("model_comparison.csv",         "Tabel metrik semua model"),
    ("predictions.csv",              "Data + hasil prediksi"),
]
for fname, desc in files:
    path = f"hasil_analisis/{fname}"
    size = os.path.getsize(path)/1024 if os.path.exists(path) else 0
    print(f"  {fname:<40} {size:>7.1f} KB  — {desc}")

print("\n✅ ANALISIS SELESAI!")

BAGIAN 1: Load & Preprocessing
Shape setelah pivot : (4188, 12)
Rentang waktu       : 2025-11-27 → 2026-05-27
Missing values      :
PM25     0
PM1      0
RH       0
Temp     0
UM003    0

Statistik deskriptif:
          PM25      PM1       RH     Temp    UM003
count  4188.00  4188.00  4188.00  4188.00  4188.00
mean     31.93    23.45    37.69    27.61   685.79
std      65.08    52.30    24.97     2.41   842.20
min       0.00     0.00     3.25    23.20     1.02
25%       5.86     5.00     6.42    25.80    10.75
50%      22.30    16.30    49.00    26.95   357.00
75%      37.00    25.90    60.00    29.00  1167.25
max     644.07   545.58    69.00    38.90  3909.66

BAGIAN 2: Exploratory Data Analysis
✅ Saved: 01_distribusi_parameter.png
✅ Saved: 02_tren_pm25_harian.png
✅ Saved: 03_pola_diurnal.png
✅ Saved: 04_pola_harian_mingguan.png
✅ Saved: 05_boxplot_waktu.png
✅ Saved: 06_tren_bulanan.png

BAGIAN 3: Analisis Korelasi
        PM25    PM1  UM003   Temp     RH
PM25   1.000  0.071  0.123 -0